# Scenario 7: Vulnerable Road User (VRU) Detection

Autonomous vehicle perception module for detecting vulnerable road users using YOLOv8.

In [3]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# 1. Load the optimized nano model for real-time processing
model = YOLO("yolov8n.pt") 

# 2. Open the video stream source
video_path = "source/test_driving_video.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Error: Could not open video file: {video_path}")
    exit()

# Extract dimensions for UI layout geometry
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Class definitions mapping
PEDESTRIAN_CLASS = 0
SPORTS_BALL_CLASS = 32
OTHER_VRU_CLASSES = [1, 3, 15, 16, 17, 18, 21, 22]

# Configure the perspective lane geometry mapping onto the street
trajectory_poly = np.array([
    [int(width * 0.12), int(height * 0.95)], 
    [int(width * 0.42), int(height * 0.55)], 
    [int(width * 0.58), int(height * 0.55)], 
    [int(width * 0.88), int(height * 0.95)]  
], np.int32)

# State Machine Tracking Variables
COOLDOWN_FRAMES = 12  
brake_timer = 0
warning_timer = 0

print("Starting Notebook Live Stream... (Press the Stop Button in Jupyter to interrupt)")

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Video stream reached the end frame.")
            break

        # AI inference over the incoming frame
        results = model(frame, verbose=False)[0]
        
        pedestrian_in_lane = False
        ball_in_lane = False

        # Draw semi-transparent cyan lane boundary layer
        overlay = frame.copy()
        cv2.fillPoly(overlay, [trajectory_poly], (255, 255, 0))
        cv2.addWeighted(overlay, 0.12, frame, 0.88, 0, frame)
        cv2.polylines(frame, [trajectory_poly], isClosed=True, color=(255, 255, 0), thickness=2)

        for box in results.boxes:
            class_id = int(box.cls[0])
            
            if class_id == PEDESTRIAN_CLASS or class_id == SPORTS_BALL_CLASS or class_id in OTHER_VRU_CLASSES:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                bottom_center_pt = (int((x1 + x2) / 2), y2)
                
                # Math Point-In-Polygon configuration check
                is_inside = cv2.pointPolygonTest(trajectory_poly, bottom_center_pt, False)

                if is_inside >= 0: 
                    if class_id == PEDESTRIAN_CLASS:
                        pedestrian_in_lane = True
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                        cv2.circle(frame, bottom_center_pt, 6, (0, 0, 255), -1)
                    elif class_id == SPORTS_BALL_CLASS:
                        ball_in_lane = True
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
                else:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)

        # State machine decay logic evaluation
        if pedestrian_in_lane:
            brake_timer = COOLDOWN_FRAMES  
            warning_timer = 0 
        elif ball_in_lane:
            if brake_timer == 0:          
                warning_timer = COOLDOWN_FRAMES
        else:
            if brake_timer > 0:   brake_timer -= 1
            if warning_timer > 0: warning_timer -= 1

        # Dynamic UI state overlay render display block
        if brake_timer > 0:
            msg = "ACTION: PEDESTRIAN DETECTED - EMERGENCY AUTOMATIC BRAKING"
            cv2.rectangle(frame, (0, height - 90), (width, height), (0, 0, 255), cv2.FILLED) 
            cv2.putText(frame, msg, (35, height - 38), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
        elif warning_timer > 0:
            msg = "WARNING: OBJECT IN PATH - PRE-CHARGING BRAKE ACTUATORS"
            cv2.rectangle(frame, (0, height - 90), (width, height), (0, 165, 255), cv2.FILLED) 
            cv2.putText(frame, msg, (35, height - 38), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
        else:
            cv2.rectangle(frame, (0, height - 90), (width, height), (30, 30, 30), cv2.FILLED) 
            cv2.putText(frame, "SYSTEM STATUS: PATH CLEAR - AUTONOMOUS CRUISING", (35, height - 38), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2, cv2.LINE_AA)

        # --- NOTEBOOK INLINE RENDERING WORKAROUND ---
        # 1. Convert frame layout from OpenCV's BGR to Matplotlib's RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # 2. Clear previous output cell completely to render new frame update
        clear_output(wait=True)
        
        # 3. Use Matplotlib to plot the matrix frame directly in the notebook
        plt.figure(figsize=(12, 7))
        plt.imshow(frame_rgb)
        plt.axis('off')  # Turn off plot coordinate grids
        plt.show()

except KeyboardInterrupt:
    print("Live stream stopped manually by user.")

finally:
    cap.release()
    print("Video loop resources closed.")

Live stream stopped manually by user.
Video loop resources closed.
